# ML - Classification (LR, DT, RF)

**Loan Approval Dataset**

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
# Load the dataset
df = pd.read_csv("loan_approval_dataset.csv")

# Remove spaces from column names and text values
df.columns = df.columns.str.strip()

for col in ["education", "self_employed", "loan_status"]:
    df[col] = df[col].astype(str).str.strip()

df


In [ ]:
# DataFrame with 7 rows
df.head(7)


In [ ]:
# Check the target values
df["loan_status"].value_counts()


In [ ]:
# Convert the target to 0 and 1
df["loan_status"] = df["loan_status"].map({
    "Rejected": 0,
    "Approved": 1
})

df["loan_status"].value_counts()


In [ ]:
# Prepare X and y
# loan_id is only an ID, so it is not used as a feature
X = df.drop(columns=["loan_status", "loan_id"])
y = df["loan_status"]


In [ ]:
# Convert categorical columns to numeric values
X = pd.get_dummies(X, columns=["education", "self_employed"], drop_first=True)
X = X.astype(float)

X.head()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred


In [ ]:
y_prob = model.predict_proba(X_test)
y_prob


In [ ]:
df_result = pd.DataFrame({
    "y_test": y_test.to_numpy(),
    "y_pred": y_pred,
    "y_prob_0_logReg": y_prob[:, 0],
    "y_prob_1_logReg": y_prob[:, 1]
})

df_result


In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

confusion_matrix(y_test, y_pred)


In [ ]:
print(classification_report(y_test, y_pred))


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))


## Logistic Regression - Ridge

In [ ]:
ridge_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=2000
)

ridge_model.fit(X_train, y_train)

y_pred_ridge = ridge_model.predict(X_test)
y_prob_ridge = ridge_model.predict_proba(X_test)

df_ridge = pd.DataFrame({
    "y_test": y_test.to_numpy(),
    "y_pred": y_pred_ridge,
    "y_prob_0_Ridge": y_prob_ridge[:, 0],
    "y_prob_1_Ridge": y_prob_ridge[:, 1]
})

df_ridge


In [ ]:
print("Ridge Accuracy:", accuracy_score(y_test, y_pred_ridge))
print(classification_report(y_test, y_pred_ridge))


## Logistic Regression - Lasso

In [ ]:
lasso_model = LogisticRegression(
    penalty="l1",
    C=1.0,
    solver="liblinear",
    max_iter=2000
)

lasso_model.fit(X_train, y_train)

y_pred_lasso = lasso_model.predict(X_test)
y_prob_lasso = lasso_model.predict_proba(X_test)

df_lasso = pd.DataFrame({
    "y_test": y_test.to_numpy(),
    "y_pred": y_pred_lasso,
    "y_prob_0_Lasso": y_prob_lasso[:, 0],
    "y_prob_1_Lasso": y_prob_lasso[:, 1]
})

df_lasso


In [ ]:
print("Lasso Accuracy:", accuracy_score(y_test, y_pred_lasso))
print(classification_report(y_test, y_pred_lasso))


## Decision Tree Classifier - Before Grid Search

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)

df_dt = pd.DataFrame({
    "y_test": y_test.to_numpy(),
    "y_pred": y_pred_dt,
    "y_prob_0_DT": y_prob_dt[:, 0],
    "y_prob_1_DT": y_prob_dt[:, 1]
})

df_dt


In [ ]:
print("Decision Tree Accuracy BEFORE Grid Search:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


## Decision Tree Classifier - After Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV

dt_param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

print("Best Parameters:", dt_grid.best_params_)
print("Best CV Score:", dt_grid.best_score_)


In [ ]:
best_dt = dt_grid.best_estimator_

y_pred_dt_grid = best_dt.predict(X_test)
y_prob_dt_grid = best_dt.predict_proba(X_test)

df_dt_grid = pd.DataFrame({
    "y_test": y_test.to_numpy(),
    "y_pred": y_pred_dt_grid,
    "y_prob_0_DT_Grid": y_prob_dt_grid[:, 0],
    "y_prob_1_DT_Grid": y_prob_dt_grid[:, 1]
})

df_dt_grid


In [ ]:
print("Decision Tree Accuracy AFTER Grid Search:", accuracy_score(y_test, y_pred_dt_grid))
print(classification_report(y_test, y_pred_dt_grid))


## Random Forest Classifier - Before Grid Search

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)

df_rf = pd.DataFrame({
    "y_test": y_test.to_numpy(),
    "y_pred": y_pred_rf,
    "y_prob_0_RF": y_prob_rf[:, 0],
    "y_prob_1_RF": y_prob_rf[:, 1]
})

df_rf


In [ ]:
print("Random Forest Accuracy BEFORE Grid Search:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


## Random Forest Classifier - After Grid Search

In [ ]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:", rf_grid.best_params_)
print("Best CV Score:", rf_grid.best_score_)


In [ ]:
best_rf = rf_grid.best_estimator_

y_pred_rf_grid = best_rf.predict(X_test)
y_prob_rf_grid = best_rf.predict_proba(X_test)

df_rf_grid = pd.DataFrame({
    "y_test": y_test.to_numpy(),
    "y_pred": y_pred_rf_grid,
    "y_prob_0_RF_Grid": y_prob_rf_grid[:, 0],
    "y_prob_1_RF_Grid": y_prob_rf_grid[:, 1]
})

df_rf_grid


In [ ]:
print("Random Forest Accuracy AFTER Grid Search:", accuracy_score(y_test, y_pred_rf_grid))
print(classification_report(y_test, y_pred_rf_grid))


## Final Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Logistic Regression - Ridge",
        "Logistic Regression - Lasso",
        "Decision Tree - Before Grid Search",
        "Decision Tree - After Grid Search",
        "Random Forest - Before Grid Search",
        "Random Forest - After Grid Search"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_ridge),
        accuracy_score(y_test, y_pred_lasso),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_dt_grid),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_rf_grid)
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_ridge),
        precision_score(y_test, y_pred_lasso),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt_grid),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf_grid)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_ridge),
        recall_score(y_test, y_pred_lasso),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_dt_grid),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf_grid)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_ridge),
        f1_score(y_test, y_pred_lasso),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_dt_grid),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf_grid)
    ]
})

comparison.sort_values("Accuracy", ascending=False)


## Confusion Matrix - Best Model

In [ ]:
best_row = comparison.loc[comparison["Accuracy"].idxmax()]
print("Best Model:", best_row["Model"])
print("Best Accuracy:", best_row["Accuracy"])


In [ ]:
predictions = {
    "Logistic Regression": y_pred,
    "Logistic Regression - Ridge": y_pred_ridge,
    "Logistic Regression - Lasso": y_pred_lasso,
    "Decision Tree - Before Grid Search": y_pred_dt,
    "Decision Tree - After Grid Search": y_pred_dt_grid,
    "Random Forest - Before Grid Search": y_pred_rf,
    "Random Forest - After Grid Search": y_pred_rf_grid
}

best_predictions = predictions[best_row["Model"]]
confusion_matrix(y_test, best_predictions)
